# Ejercicio: Web Scraping

## Nombre: Edison Quizhpe

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

rag_corpus
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [53]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [54]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [55]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [56]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [57]:
# Buscamos, filtramos, reparamos la URL y quitamos duplicados
recipe_urls = list(dict.fromkeys(
    ("https://www.allrecipes.com" + a["href"] if a["href"].startswith("/") else a["href"]).split("?")[0]
    for a in soup.find_all("a", href=True) 
    if "/recipe/" in a["href"] and "/recipes/" not in a["href"] and "authentication" not in a["href"]
))

print(f"Se encontraron {len(recipe_urls)} recetas reales:")
for url in recipe_urls:
    print(url)

Se encontraron 16 recetas reales:
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.com/recipe/89

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [58]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# Asumimos que 'recipe_urls' ya existe en la memoria de tu notebook por la celda anterior
corpus = []
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126.0"}

print(f"Iniciando descarga profunda de {len(recipe_urls)} recetas...")

for url in recipe_urls:
    try:
        respuesta = requests.get(url, headers=headers, timeout=10)
        
        if respuesta.status_code != 200:
            print(f"⚠️ Ignorada (Error {respuesta.status_code}): {url}")
            continue

        soup = BeautifulSoup(respuesta.text, "html.parser")
        titulo_tag = soup.find("h1")
        
        if not titulo_tag:
            continue # Saltamos si no hay título

        # Extraemos todo
        titulo = titulo_tag.text.strip()
        descripcion = (soup.find("meta", {"name": "description"}) or {}).get("content", "").strip()
        ingredientes = [li.text.strip() for li in soup.find_all("li", class_=re.compile("ingredients"))]
        instrucciones = [p.text.strip() for p in soup.find_all("p", class_=re.compile("mntl-sc-block-html"))]
        nutricion = [s.parent.text.strip() for s in soup.find_all("span", class_=re.compile("nutrient-name"))]

        # Si tiene los datos clave, la guardamos
        if ingredientes and instrucciones:
            texto_rag = (
                f"Título: {titulo}\n"
                f"Descripción: {descripcion}\n"
                f"Ingredientes:\n" + "\n".join(f"- {i}" for i in ingredientes) + "\n"
                f"Instrucciones:\n" + "\n".join(f"{idx+1}. {paso}" for idx, paso in enumerate(instrucciones)) + "\n"
                f"Nutrición:\n" + ", ".join(nutricion)
            )
            
            corpus.append({
                "url": url,
                "titulo": titulo,
                "texto_rag": texto_rag
            })
            print(f"✅ Extraída con éxito: {titulo}")

        # Evitar bloqueos
        time.sleep(1)

    except Exception as e:
        print(f"⚠️ Error técnico en {url}: {e}")

# Guardamos el CSV final
df = pd.DataFrame(corpus)
df.to_csv("mi_corpus_allrecipes.csv", index=False, encoding="utf-8")

print(f"\n¡Proceso completado! Tienes {len(corpus)} recetas guardadas y listas para conectar a Gemini.")

Iniciando descarga profunda de 16 recetas...
✅ Extraída con éxito: Cilantro-Lime Grilled Chicken
✅ Extraída con éxito: Buttermilk Barbecue Chicken
✅ Extraída con éxito: Grilled Spatchcocked Chicken
✅ Extraída con éxito: Beer Butt Chicken
✅ Extraída con éxito: Good Frickin’ Paprika Chicken
✅ Extraída con éxito: Miso Honey Chicken
✅ Extraída con éxito: Rosemary Buttermilk Chicken
✅ Extraída con éxito: Smoked Beer Butt Chicken
✅ Extraída con éxito: The Best Beer Can Chicken Ever
✅ Extraída con éxito: Best Beer Can Chicken
✅ Extraída con éxito: Drunk Chicken
✅ Extraída con éxito: Grilled Chicken Under a Brick
✅ Extraída con éxito: Smoked Whole Chicken
✅ Extraída con éxito: Easy Barbeque Chicken
✅ Extraída con éxito: Darn Good Chicken
✅ Extraída con éxito: Beer Can Chicken

¡Proceso completado! Tienes 16 recetas guardadas y listas para conectar a Gemini.


In [59]:
import pandas as pd
import json
from pathlib import Path

# 1. Definimos corpus_dir basándonos en la carpeta actual
notebook_dir = Path.cwd()
corpus_dir = notebook_dir / "rag_corpus"

# Crea la carpeta "rag_corpus"
corpus_dir.mkdir(parents=True, exist_ok=True)

# 2. Definimos las rutas de los archivos
corpus_jsonl_path = corpus_dir / "recipes_corpus.jsonl"
corpus_csv_path = corpus_dir / "recipes_corpus.csv"

# 3. Creamos el DataFrame
corpus_df = pd.DataFrame(corpus)

# 4. Guardamos como JSONL
with corpus_jsonl_path.open("w", encoding="utf-8") as jsonl_file:
    for record in corpus:
        jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")

# 5. Guardamos como CSV
corpus_df.to_csv(corpus_csv_path, index=False, encoding="utf-8")

print(f"✅ Datos guardados correctamente en:\n- {corpus_jsonl_path}\n- {corpus_csv_path}")

✅ Datos guardados correctamente en:
- c:\Users\wwwed\OneDrive - Escuela Politécnica Nacional\Escritorio\EPN\SEPTIMO SEMESTRE\ir26a\12webcrawling\rag_corpus\recipes_corpus.jsonl
- c:\Users\wwwed\OneDrive - Escuela Politécnica Nacional\Escritorio\EPN\SEPTIMO SEMESTRE\ir26a\12webcrawling\rag_corpus\recipes_corpus.csv


In [62]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from IPython.display import display, Markdown

# 1. CARGA DIRECTA DEL CSV
try:
    corpus_df = pd.read_csv("mi_corpus_allrecipes.csv")
    corpus_df = corpus_df.fillna("") # Protege contra celdas vacías
    print(f"Datos cargados correctamente: {len(corpus_df)} recetas encontradas.\n")
except FileNotFoundError:
    raise FileNotFoundError("❌ No se encontró 'mi_corpus_allrecipes.csv'. Asegúrate de correr el scraper primero.")

# 2. VECTORIZACIÓN (EMBEDDINGS)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Vectorizando recetas...")
doc_embeddings = model.encode(
    corpus_df["texto_rag"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False,
)

# 3. FUNCIONES DE BÚSQUEDA Y CONTEXTO
def rag_search(query, top_k=3):
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    ranked_indices = scores.argsort()[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(ranked_indices, 1):
        row = corpus_df.iloc[idx]
        results.append({
            "rank": rank,
            "titulo": row.get("titulo", ""),
            "url": row.get("url", ""),
            "score": float(scores[idx]),
            "texto_rag": row.get("texto_rag", "") 
        })
    return results

def build_gemini_context(results):
    context_lines = []
    for item in results:
        context_lines.append(
            f"[{item['rank']}] {item['titulo']} | similitud={item['score']:.3f}\n"
            f"URL: {item['url']}\n"
            f"{item['texto_rag']}\n"
            f"{'-'*40}"
        )
    return "\n\n".join(context_lines)

# 4. PRUEBA DE BÚSQUEDA Y VISUALIZACIÓN
query = "beer"
retrieved_results = rag_search(query, top_k=3)

# Generamos el contexto crudo para la IA (oculto en memoria)
context = build_gemini_context(retrieved_results)

# Imprimimos los resultados para el usuario
display(Markdown(f"Resultados semánticos para: *'{query}'*"))
display(Markdown("---"))

for item in retrieved_results:
    # Maquillamos el texto solo para la pantalla
    texto_visual = item['texto_rag']
    texto_visual = texto_visual.replace("Título:", "Título:")
    texto_visual = texto_visual.replace("Descripción:", "\n> Descripción:")
    texto_visual = texto_visual.replace("Ingredientes:", "\n Ingredientes:")
    texto_visual = texto_visual.replace("Instrucciones:", "\nInstrucciones:")
    texto_visual = texto_visual.replace("Nutrición:", "\nNutrición:")
    
    tarjeta = (
        f"### {item['rank']}. [{item['titulo']}]({item['url']})\n"
        f"*Nivel de similitud: {item['score']:.3f}*\n\n"
        f"{texto_visual}\n\n"
        f"---"
    )
    display(Markdown(tarjeta))

Datos cargados correctamente: 16 recetas encontradas.



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9200.97it/s]


Vectorizando recetas...


Resultados semánticos para: *'beer'*

---

### 1. [Drunk Chicken](https://www.allrecipes.com/recipe/19944/drunk-chicken/)
*Nivel de similitud: 0.306*

Título: Drunk Chicken

> Descripción: Drunken chicken is cooked over the grill with a beer can up inside for a fun and easy dish!

 Ingredientes:
- 1 (2 to 3 pound) whole chicken
- 1 (12 fluid ounce) can beer
- 5 tablespoons poultry seasoning
- 4 dashes liquid smoke flavoring
- 4  bay leaves
- 1 long metal skewer

Instrucciones:
1. Rinse and dry chicken. Remove excess fat and leave skin on. Lift skin from breast and thigh areas; slide bay leaves under skin. Coat chicken with poultry seasoning.
2. Drink 1/2 of the can of beer; pour liquid smoke into remaining beer. Raise tab on beer can until it is in the straight-up position.
3. Insert beer can into chicken from the bottom until even with bottom of chicken. Insert skewer through wing, ribs, beer can tab, and out the opposite side. (This keeps the can from falling out of chicken.)
4. Preheat the grill by lighting the coals. Spread the coals to form a ring around the outside edge of the grill.
5. Place chicken in the center, standing up on the can to cook. Close grill; cook for 2 hours. A meat thermometer inserted into the center should read at least 165 degrees F (74 degrees C).
6. Remove chicken carefully from the grill so as not to spill the contents of the can. Remove skewer and beer can; let chicken sit for 15 minutes before cutting.

Nutrición:
Total Carbohydrate
6g, Dietary Fiber
1g, Total Sugars
0g, Protein
44g, Total Fat
23g, Saturated Fat
6g, Cholesterol
140mg, Vitamin C
1mg, Sodium
136mg, Calcium
75mg, Iron
4mg, Potassium
412mg

---

### 2. [The Best Beer Can Chicken Ever](https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/)
*Nivel de similitud: 0.305*

Título: The Best Beer Can Chicken Ever

> Descripción: A whole grilled chicken gets a burst of flavor from a warm and complex spice rub and a can of dark stout beer with hot peppers and garlic cloves.

 Ingredientes:
- 1 cup chocolate stout beer
- 3  green Thai chile peppers
- 3 cloves garlic, peeled
- 3 tablespoons brown sugar
- 2 teaspoons dry mustard
- 1 teaspoon garam masala
- 1 teaspoon kosher salt
- 1 teaspoon ground black pepper
- ½ teaspoon cayenne pepper
- ½ teaspoon ground cumin
- ½ teaspoon ground cinnamon
- ½ teaspoon onion powder
- ½ teaspoon garlic powder
- ¼ teaspoon ground nutmeg
- 1 (5 pound) whole chicken

Instrucciones:
1. Preheat grill for medium heat. If using charcoal, push coals to the side of the grilling area for indirect heat.
2. Pour stout beer into an empty 12-ounce soda can and drop Thai chilies and garlic cloves into the can. Mix brown sugar, dry mustard, garam masala, kosher salt, black pepper, cayenne pepper, cumin, cinnamon, onion powder, garlic powder, and nutmeg in a bowl.
3. Rinse the chicken and coat the skin with the entire batch of spice rub. Place soda can containing beer mixture onto the prepared grill and sit the chicken upright onto the can.
4. Grill chicken over indirect heat until juices run clear and an instant-read meat thermometer inserted into the thickest part of the breast, not touching bone, reads at least 165 degrees F (75 degrees C), about 45 minutes. Let chicken rest 10 minutes before serving.

Nutrición:
Total Carbohydrate
10g, Dietary Fiber
1g, Total Sugars
7g, Protein
52g, Total Fat
29g, Saturated Fat
8g, Cholesterol
162mg, Vitamin C
1mg, Sodium
480mg, Calcium
45mg, Iron
3mg, Potassium
484mg

---

### 3. [Beer Butt Chicken](https://www.allrecipes.com/recipe/14531/beer-butt-chicken/)
*Nivel de similitud: 0.274*

Título: Beer Butt Chicken

> Descripción: This beer butter chicken recipe combines beer, butter, chicken, and seasonings for a moist and flavorful meal that will have people talking.

 Ingredientes:
- 1 cup butter, divided
- 2 tablespoons garlic salt, divided
- 2 tablespoons paprika, divided
- salt and pepper to taste
- 1 (12 fluid ounce) can beer
- 1 (4 pound) whole chicken

Instrucciones:
1. Preheat an outdoor grill for low heat and lightly oil the grate.
2. Melt 1/2 cup butter in a small skillet. Mix in 1 tablespoon garlic salt, 1 tablespoon paprika, salt, and pepper.
3. Discard 1/2 of the beer, leaving the remainder in the can. Add remaining butter, garlic salt, paprika, salt, and pepper to the beer can. Place the can on a baking sheet or disposable pan. Set chicken upright on the beer can, inserting the can into the cavity of the chicken. Baste chicken with melted, seasoned butter.
4. Place the baking sheet with beer and chicken on the preheated grill. Cook over low heat until no longer pink at the bone and the juices run clear, about 3 hours. An instant-read thermometer inserted into the thickest part of the thigh, near the bone, should read 165 degrees F (74 degrees C).
5. Remove from the grill, cover with a doubled sheet of aluminum foil, and allow to rest in a warm area for 10 minutes before slicing.

Nutrición:
Total Carbohydrate
3g, Dietary Fiber
1g, Total Sugars
0g, Protein
31g, Total Fat
40g, Saturated Fat
19g, Cholesterol
158mg, Vitamin C
1mg, Sodium
1618mg, Calcium
29mg, Iron
2mg, Potassium
335mg

---

In [67]:
from google import genai
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Función principal de consulta RAG
def ask_gemini_about_recipes(query, retrieved_results, model_name="gemini-2.5-flash"):
    load_dotenv()
    api_key = os.getenv("api_key") 
    
    context = build_gemini_context(retrieved_results)
    
    prompt = f"""
Eres un asistente experto en recetas de cocina.
Responde en español, usando UNICAMENTE el contexto recuperado del corpus local.
Si el contexto no contiene la respuesta o no basta para responder, indícalo explícitamente.

Contexto recuperado:
{context}

Pregunta del usuario:
{query}

Instrucciones:
1. Responde de forma clara, directa y breve.
2. Cita las recetas más relevantes por su nombre exacto.
3. Si hay varias opciones, compara brevemente sus similitudes y sugiere la mejor opción.
""".strip()

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(model=model_name, contents=prompt)
    return response.text

# Ejecución del pipeline
gemini_response = ask_gemini_about_recipes(query, retrieved_results)

display(Markdown("Respuesta de Gemini (Asistente RAG):"))
display(Markdown(gemini_response))

Respuesta de Gemini (Asistente RAG):

Las siguientes recetas utilizan cerveza:

*   **Drunk Chicken:** Utiliza una lata de cerveza de 12 onzas líquidas. Se bebe la mitad de la cerveza, se vierte saborizante de humo líquido en el resto, y la lata se inserta en el pollo.
*   **The Best Beer Can Chicken Ever:** Utiliza 1 taza de cerveza stout de chocolate. Se vierte en una lata de refresco vacía de 12 onzas, y se añaden chiles tailandeses verdes y dientes de ajo a la lata. Luego se coloca el pollo sobre la lata.
*   **Beer Butt Chicken:** Utiliza una lata de cerveza de 12 onzas líquidas. Se desecha la mitad de la cerveza, y se añaden mantequilla, sal de ajo, pimentón, sal y pimienta a la cerveza restante en la lata. La lata se inserta en la cavidad del pollo.

**Comparación y sugerencia:**
Todas las recetas utilizan cerveza insertada en la cavidad del pollo para cocinarlo en la parrilla.
*   **Drunk Chicken** es para un plato divertido y fácil con un toque ahumado.
*   **Beer Butt Chicken** promete una comida húmeda y sabrosa con mantequilla y condimentos en la cerveza.
*   **The Best Beer Can Chicken Ever** ofrece una explosión de sabor con una cerveza stout oscura, chiles picantes y dientes de ajo dentro de la lata, además de un complejo aderezo de especias en el pollo.

Si buscas la opción con el perfil de sabor más complejo y un "estallido de sabor" adicional de la cerveza y sus adiciones, **The Best Beer Can Chicken Ever** podría ser la mejor opción. Si prefieres un sabor más simple y ahumado, **Drunk Chicken** es adecuada. Para un sabor rico y mantecoso, **Beer Butt Chicken** es la indicada.